# FiremeX — retrain the fire model for indoor CCTV (Google Colab)

Our model (model-v2, trained on D-Fire) missed thin smoke and small fires on our
own EZVIZ camera. This notebook teaches it indoor fires, **without forgetting**
what it already knows, using three sets of pictures:

| Pictures | What | Why |
|---|---|---|
| **Home fire dataset** (6,500) | indoor home fires, small flames, early smoke, surveillance views | the main new knowledge |
| **D-Fire** (10,000 of them) | the photos model-v2 learned from | reminders, so it does not forget |
| **Our own camera** (239) | the bowl fire and lighter clips we recorded | our room and camera colours |

1. **Runtime → Change runtime type → T4 GPU**, then Save.
2. **Runtime → Run all.** When it asks, upload **`firemex_own_dataset.zip`**
   (in the project at `ml/datasets/`). About 1½–2 hours. Keep the tab open.
3. Read the comparison table near the end. The last cell downloads
   **`fire_model_v3.pt`** only if the new model beats model-v2.

Datasets: Home Fire Dataset by Pengbo (github.com/PengBo0/Home-fire-dataset,
CC BY-NC 4.0 — research and study use only) and D-Fire by Gaia
(github.com/gaiasd/DFireDataset). Please credit both in the report.

In [ ]:
# The add-on runs Ultralytics 8.2.0. Training with the same version means the
# saved model loads in the add-on exactly as it does here.
!pip -q install ultralytics==8.2.0 kagglehub
# Removed BEFORE ultralytics is first imported:
#  - wandb: Ultralytics would log to it, and it rejects our run folder.
#  - dill: Ultralytics saves with dill when it is installed, and the add-on
#    cannot load a dill-pickled model (our first Colab model had to be fixed by hand).
!pip -q uninstall -y wandb dill
import os
# Colab's PyTorch refuses to load YOLO model files by default
# ("Weights only load failed"). These are Ultralytics' own files.
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
import ultralytics, torch
print("ultralytics", ultralytics.__version__, "| GPU:",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch the runtime to T4 GPU")

In [ ]:
# 1) Home fire dataset, straight from its GitHub release (no sign-in needed).
#    Its classes are 0 = flame, 1 = smoke. FiremeX uses 0 = smoke, 1 = fire,
#    so every label is swapped here - checked by drawing boxes of each class.
import glob, os, shutil
BASE = "https://github.com/PengBo0/Home-fire-dataset/releases/download/v1.0.0"
os.makedirs("/content/homefire", exist_ok=True)
for split in ("train", "val", "test"):
    if not os.path.exists(f"/content/homefire/{split}/images"):
        !wget -q -O /content/{split}.zip {BASE}/{split}.zip
        !unzip -q -o /content/{split}.zip -d /content/homefire/{split}
        os.remove(f"/content/{split}.zip")
SWAP = {"0": "1", "1": "0"}
if not os.path.exists("/content/homefire/.swapped"):     # running this cell twice must not swap back
    for lab in glob.glob("/content/homefire/*/labels/*.txt"):
        lines = [l.split() for l in open(lab) if l.strip()]
        with open(lab, "w") as fh:
            for l in lines:
                fh.write(" ".join([SWAP[l[0]]] + l[1:]) + "\n")
    open("/content/homefire/.swapped", "w").close()
for split in ("train", "val", "test"):
    print(split, len(glob.glob(f"/content/homefire/{split}/images/*")), "pictures")

In [ ]:
# 2) D-Fire reminders: 10,000 random training photos (fixed seed). Classes are
#    already 0 = smoke, 1 = fire. If Kaggle asks you to sign in:
#    kaggle.com -> Settings -> Create New Token, then upload kaggle.json with
#    the folder icon on the left and run this cell again.
#    If it still fails, set USE_DFIRE = False - training works without it,
#    there is just more risk of forgetting outdoor fires.
USE_DFIRE = True
import random
dfire_train = dfire_test = None
if USE_DFIRE:
    import kagglehub
    root = kagglehub.dataset_download("sayedgamal99/smoke-fire-detection-yolo")
    def split_dir(*names):
        for n in names:
            hits = glob.glob(f"{root}/**/{n}/images", recursive=True)
            if hits:
                return hits[0]
    src_train, dfire_test = split_dir("train"), split_dir("test")
    images = sorted(glob.glob(f"{src_train}/*"))
    random.Random(0).shuffle(images)
    dfire_train = "/content/dfire_subset/images"
    os.makedirs(dfire_train, exist_ok=True)
    os.makedirs("/content/dfire_subset/labels", exist_ok=True)
    for img in images[:10000]:
        stem = os.path.splitext(os.path.basename(img))[0]
        shutil.copy(img, dfire_train)
        lab = os.path.join(os.path.dirname(src_train), "labels", stem + ".txt")
        if os.path.exists(lab):
            shutil.copy(lab, "/content/dfire_subset/labels")
    print("D-Fire reminders:", len(os.listdir(dfire_train)), "| D-Fire test:", dfire_test)

In [ ]:
# 3) Our own camera pictures (bowl fire, lighter, room). Upload
#    firemex_own_dataset.zip when the button appears. Its test clips
#    (take3_02, lighter_1m_03) are NEVER trained on - they are how we judge.
from google.colab import files
if not os.path.exists("/content/own/train/images"):
    up = files.upload()
    name = next(iter(up))
    !unzip -q -o "/content/{name}" -d /content/own
print("own train:", len(glob.glob("/content/own/train/images/*")),
      "| own test:", len(glob.glob("/content/own/test/images/*")))

In [ ]:
# 4) One training list from all three. Validation (used to pick the best epoch)
#    is the Home fire val split; each test set is scored separately below.
import yaml
train_dirs = ["/content/homefire/train/images", "/content/own/train/images"]
if dfire_train:
    train_dirs.append(dfire_train)
names = {0: "smoke", 1: "fire"}
yaml.safe_dump({"train": train_dirs, "val": "/content/homefire/val/images",
                "test": "/content/homefire/test/images", "names": names},
               open("/content/mix.yaml", "w"))
tests = {"Home fire test": "/content/homefire/test/images",
         "Our camera test": "/content/own/test/images"}
if dfire_test:
    tests["D-Fire test"] = dfire_test
for label, path in tests.items():
    yaml.safe_dump({"train": path, "val": path, "names": names},
                   open(f"/content/test_{label.split()[0].lower()}.yaml", "w"))
print(open("/content/mix.yaml").read())

In [ ]:
# 5) Start from model-v2 (what the add-on runs today), not from scratch, so it
#    keeps what it learned from D-Fire. ~40 epochs on a T4: about 1-1.5 hours.
!wget -q -O /content/model_v2.pt https://github.com/malika1234m/Firemax/releases/download/model-v2/fire_model.pt
from ultralytics import YOLO
model = YOLO("/content/model_v2.pt")
model.train(data="/content/mix.yaml", epochs=40, imgsz=640, batch=32,
            patience=10, seed=0, deterministic=True, workers=2,
            hsv_h=0.005,          # flame colour matters - keep hue jitter tiny
            project="/content/runs", name="v3", exist_ok=True)

In [ ]:
# 6) Old vs new on every test set, scored separately from training (in an
#    earlier FiremeX run the in-training scores were wrong, so we check).
#    Recall = share of real fire/smoke the model finds; it matters most here.
import pandas as pd
def score(weights, label):
    r = YOLO(weights).val(data=f"/content/test_{label.split()[0].lower()}.yaml", split="val",
                         imgsz=640, batch=32, plots=False, verbose=False)
    per = {r.names[int(i)]: float(r.box.r[k]) for k, i in enumerate(r.box.ap_class_index)}
    return {"mAP50": round(float(r.box.map50), 3),
            "fire recall": round(per.get("fire", float("nan")), 3),
            "smoke recall": round(per.get("smoke", float("nan")), 3)}
candidates = {"model-v2 (now)": "/content/model_v2.pt",
              "new best.pt": "/content/runs/v3/weights/best.pt",
              "new last.pt": "/content/runs/v3/weights/last.pt"}
rows = []
for name, w in candidates.items():
    for label in tests:
        rows.append({"model": name, "test set": label, **score(w, label)})
table = pd.DataFrame(rows)
display(table.pivot(index="model", columns="test set", values="mAP50"))
display(table)

In [ ]:
# 7) Pick the new checkpoint with the best average mAP50 over all test sets,
#    and only download it if it beats model-v2 on average AND is not much
#    worse (more than 0.05) on any single test set - one better number is not
#    enough if it broke something else.
avg = table.groupby("model")["mAP50"].mean()
new = max(["new best.pt", "new last.pt"], key=lambda m: avg[m])
old_by = table[table.model == "model-v2 (now)"].set_index("test set")["mAP50"]
new_by = table[table.model == new].set_index("test set")["mAP50"]
worse = {t: (old_by[t], new_by[t]) for t in old_by.index if new_by[t] < old_by[t] - 0.05}
print(f"average mAP50: model-v2 {avg['model-v2 (now)']:.3f}  |  {new} {avg[new]:.3f}")
if avg[new] > avg["model-v2 (now)"] and not worse:
    shutil.copy(candidates[new], "/content/fire_model_v3.pt")
    !sha256sum /content/fire_model_v3.pt
    files.download("/content/fire_model_v3.pt")
    print("Better - downloading fire_model_v3.pt")
else:
    print("NOT better, keep model-v2. Worse on:", worse or "average")